# Phase 4 — Continuous-Time Markov Chains and Birth-Death Processes

Companion notebook to `notes/phase4-continuous-time.md`. Constructs the M/M/$\infty$ headcount model, verifies its Poisson stationary distribution, computes the expected trajectory and 5–95% band, and answers $P(X_{12} \geq 50 \mid X_0 = 45)$.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import poisson

from src.ctmc import BirthDeathProcess, CTMC

np.set_printoptions(precision=4, suppress=True)
sns.set_style("white")

## 1. Build the M/M/$\infty$ chain

$\lambda = 2$/month, $\mu_n = n \cdot 0.025$, truncate at $n_{\max} = 200$.

In [ ]:
LAMBDA, MU, N_MAX = 2.0, 0.025, 200
bd = BirthDeathProcess.from_constant_hiring(LAMBDA, MU, n_max=N_MAX)
rho = LAMBDA / MU
print(f'rho = {rho:.0f}')
print('Q[44:48, 44:48] =\n', bd.Q[44:48, 44:48])

## 2. Stationary distribution: detailed balance vs Poisson

In [ ]:
pi = bd.stationary_birth_death()
expected = poisson.pmf(np.arange(N_MAX + 1), mu=rho)
print(f'max abs diff vs Poisson({rho:.0f}): {np.abs(pi - expected).max():.2e}')
print(f'mean   = {(np.arange(N_MAX + 1) * pi).sum():.4f}  (should be {rho})')
print(f'var    = {((np.arange(N_MAX + 1) - rho) ** 2 * pi).sum():.4f}  (should be {rho})')

## 3. Expected trajectory: closed form vs $e^{Qt}$

In [ ]:
n0 = 45
times = np.array([0, 6, 12, 24, 50, 100])
states = np.arange(N_MAX + 1)

means_expm = []
for t in times:
    row = bd.transition_matrix(t)[n0] if t > 0 else np.eye(N_MAX + 1)[n0]
    means_expm.append((states * row).sum())

df = pd.DataFrame(
    {
        'closed-form': bd.expected_trajectory(n0=n0, times=times),
        'from exp(Qt)': means_expm,
    },
    index=pd.Index(times, name='t (months)'),
)
df.round(4)

## 4. The headline question — $P(X_{12} \geq 50 \mid X_0 = 45)$

In [ ]:
P12 = bd.transition_matrix(12.0)
row = P12[45]
prob = row[50:].sum()
print(f'P(X_12 >= 50 | X_0 = 45) = {prob:.4f}')
print(f'E[X_12 | X_0 = 45]       = {(states * row).sum():.2f}')
# Approximate Normal(rho + (n0-rho)e^{-mu t}, ...): variance is harder to
# compute analytically here; we just read it off the exact distribution.
var_exact = ((states - (states * row).sum()) ** 2 * row).sum()
print(f'Var[X_12 | X_0 = 45]     = {var_exact:.2f}')

## 5. Visualise stationary + trajectory

In [ ]:
fig, (ax_pi, ax_traj) = plt.subplots(1, 2, figsize=(13, 4.5))

n_plot = np.arange(0, 130)
ax_pi.bar(n_plot, pi[n_plot], color='#1D76DB', alpha=0.65, edgecolor='white',
          label='$\\pi_n$')
ax_pi.plot(n_plot, poisson.pmf(n_plot, mu=rho), color='#B60205', lw=1.8,
           label=f'Poisson({rho:.0f})')
ax_pi.set_xlabel('team size $n$')
ax_pi.set_ylabel('$\\pi_n$')
ax_pi.set_title('Stationary distribution')
ax_pi.legend()

t_grid = np.linspace(0, 60, 61)
exp_traj = bd.expected_trajectory(n0=n0, times=t_grid)
ax_traj.plot(t_grid, exp_traj, color='#1D76DB', lw=2)
ax_traj.axhline(rho, color='black', ls=':', alpha=0.6, label=f'$\\rho$ = {rho:.0f}')
ax_traj.scatter([0], [n0], color='black', zorder=5, label=f'$n_0$ = {n0}')
ax_traj.set_xlabel('$t$ (months)')
ax_traj.set_ylabel('$\\mathbb{E}[n(t)]$')
ax_traj.set_title('Expected trajectory (ODE)')
ax_traj.legend()

fig.tight_layout()

## 6. Two-state CTMC sanity check

For Q = [[-α, α], [β, -β]] the stationary is (β/(α+β), α/(α+β)) and
$P(t) = e^{Qt}$ has a known closed form. Quick sanity check.

In [ ]:
alpha, beta = 1.0, 2.0
Q = np.array([[-alpha, alpha], [beta, -beta]])
ctmc2 = CTMC(Q)
pi2 = ctmc2.stationary()
expected2 = np.array([beta / (alpha + beta), alpha / (alpha + beta)])
print(f'pi (numerical) = {pi2}')
print(f'pi (analytical) = {expected2}')
print(f'P(t=2) =\n{ctmc2.transition_matrix(2.0)}')